# CPSC 452 Final Project

## Conservation-Guided Diffusion (Trajectory Physics)

This notebook runs the project pipeline end-to-end for **Day 1 (data)** and **Day 2 (CDN)**:
- Generate/load projectile and pendulum trajectories
- Verify data scaling
- Train the Conservation Discovery Network (CDN)
- Save validation scatter plots to `figures/`

## Setup

In [1]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# Build absolute project paths (works in Jupyter or nbconvert)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "figures")

# Ensure imports work when running from notebooks/
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Models dir:", MODELS_DIR)
print("Figures dir:", FIGURES_DIR)
print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Project root: C:\Users\andre\Documents\452finalproject
Data dir: C:\Users\andre\Documents\452finalproject\data
Models dir: C:\Users\andre\Documents\452finalproject\models
Figures dir: C:\Users\andre\Documents\452finalproject\figures
Torch: 2.8.0+cpu
Device: cpu


## Day 1 — Data generation (optional) + load

In [2]:
from src.data_generation.projectile import generate_projectile_data
from src.data_generation.pendulum import generate_pendulum_data

# Toggle to regenerate .npy files (otherwise we just load existing)
REGENERATE_DATA = False

proj_path = os.path.join(DATA_DIR, "projectile", "trajectories.npy")
pend_path = os.path.join(DATA_DIR, "pendulum", "trajectories.npy")

if REGENERATE_DATA:
    os.makedirs(os.path.join(DATA_DIR, "projectile"), exist_ok=True)
    os.makedirs(os.path.join(DATA_DIR, "pendulum"), exist_ok=True)

    proj = generate_projectile_data()
    np.save(proj_path, proj)

    pend = generate_pendulum_data()
    np.save(pend_path, pend)

proj_raw = np.load(proj_path)
pend_raw = np.load(pend_path)

print("Projectile:", proj_raw.shape, proj_raw.dtype)
print("Pendulum:", pend_raw.shape, pend_raw.dtype)

Projectile: (10000, 100, 4) float64
Pendulum: (10000, 100, 2) float64


## Scaling + sanity checks

- Projectile uses **min-max to [0,1]**
- Pendulum uses **standardization (mean 0, std 1)**

In [3]:
from src.data_generation.utils import scale_trajectories, unscale_trajectories
from src.data_generation.projectile import compute_energy_projectile
from src.data_generation.pendulum import compute_energy_pendulum

proj_scaled, proj_stats = scale_trajectories(proj_raw, mode="minmax01")
pend_scaled, pend_stats = scale_trajectories(pend_raw, mode="standardize")

print("[projectile] scaled min:", proj_scaled.reshape(-1, 4).min(axis=0))
print("[projectile] scaled max:", proj_scaled.reshape(-1, 4).max(axis=0))

print("[pendulum] scaled mean:", pend_scaled.reshape(-1, 2).mean(axis=0))
print("[pendulum] scaled std:", pend_scaled.reshape(-1, 2).std(axis=0))

# Energy conservation checks in raw space
E_proj = compute_energy_projectile(proj_raw)
E_pend = compute_energy_pendulum(pend_raw)
print("Projectile mean within-trajectory energy std:", float(E_proj.std(axis=1).mean()))
print("Pendulum mean within-trajectory energy std:", float(E_pend.std(axis=1).mean()))

[projectile] scaled min: [0. 0. 0. 0.]


[projectile] scaled max: [1. 1. 0. 1.]
[pendulum] scaled mean: [2.20557739e-16 1.26545885e-16]


[pendulum] scaled std: [1. 1.]


Projectile mean within-trajectory energy std: 2.9546709849069815e-14
Pendulum mean within-trajectory energy std: 2.89426460877547e-10


## Day 2 — Train CDN + validate (scatter plot vs analytical energy)

In [4]:
from src.training.train_cdn import CDNTrainConfig, train_cdn
from src.evaluation.validate_cdn import validate_cdn

# Ensure outputs go to the project folders even when executed from notebooks/
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.chdir(PROJECT_ROOT)


def run_cdn_for_env(env_name: str, raw: np.ndarray, scaled: np.ndarray, stats, state_dim: int, scaling_mode: str):
    # Mild energy alignment to avoid learning arbitrary monotone transforms (esp. pendulum)
    energy0 = None
    cfg = CDNTrainConfig(
        env_name=env_name,
        state_dim=state_dim,
        epochs=80,
        batch_size=256,
        lr=1e-3,
        lambda_var=0.1,
        epsilon=1.0,
        var_reg="hinge",
        grad_clip=1.0,
        log_grad_norm=False,
        save_dir=MODELS_DIR,
    )

    if env_name == "pendulum":
        denorm_train = unscale_trajectories(scaled, stats, mode=scaling_mode)
        energy0 = compute_energy_pendulum(denorm_train)[:, 0]
        cfg.lambda_align = 0.2

    model, _hist = train_cdn(scaled, cfg, energy0_np=energy0)

    if env_name == "projectile":
        energy_fn = lambda t: compute_energy_projectile(unscale_trajectories(t, stats, mode=scaling_mode))
    else:
        energy_fn = lambda t: compute_energy_pendulum(unscale_trajectories(t, stats, mode=scaling_mode))

    r2 = validate_cdn(model, scaled, energy_fn, env_name=env_name)
    return r2

r2_proj = run_cdn_for_env("projectile", proj_raw, proj_scaled, proj_stats, state_dim=4, scaling_mode="minmax01")
r2_pend = run_cdn_for_env("pendulum", pend_raw, pend_scaled, pend_stats, state_dim=2, scaling_mode="standardize")

print("Final R^2 projectile:", r2_proj)
print("Final R^2 pendulum:", r2_pend)

Training CDN (projectile) on cpu


Epoch   1/80 Loss: 0.046445  Consistency: 0.000848  Variance: 8.3305


Epoch  10/80 Loss: 0.000000  Consistency: 0.000000  Variance: 34.3033


Epoch  20/80 Loss: 0.000000  Consistency: 0.000000  Variance: 34.1443


Epoch  30/80 Loss: 0.000000  Consistency: 0.000000  Variance: 34.0321


...

Epoch  50/80 Loss: 0.000002  Consistency: 0.000001  Variance: 6.2365


Epoch  60/80 Loss: 0.000002  Consistency: 0.000001  Variance: 6.1604


Epoch  70/80 Loss: 0.000002  Consistency: 0.000001  Variance: 6.1133


Epoch  80/80 Loss: 0.000002  Consistency: 0.000001  Variance: 6.1039


[pendulum] Pearson r = 1.0000, R^2 = 1.0000
Saved figures/cdn_validation_pendulum.png
Final R^2 projectile: 0.9723096068212291
Final R^2 pendulum: 0.9999511276005194


## Outputs

After running the notebook, you should have:
- `figures/cdn_validation_projectile.png`
- `figures/cdn_validation_pendulum.png`
- `models/cdn_projectile_best.pt`
- `models/cdn_pendulum_best.pt`